# YOLOComVis: Pipeline Fish Vision Mandiri

Notebook ini menyiapkan pipeline deteksi ikan, deteksi lesi, dan klasifikasi penyakit dari awal sampai laporan akhir.

## Sumber data

- **Fish4Knowledge**: arsip resmi Fish Recognition Ground-Truth.
- **FishDisease**: tiga arsip resmi dari halaman CVL JNU: gambar, anotasi, dan pathogen types.
- **Freshwater Fish Disease**: endpoint download API publik Kaggle.
- **Pipeline**: script diambil dari repository GitHub proyek secara otomatis.

## Urutan eksekusi

1. Instal dependensi dan konfigurasi runtime.
2. Unduh pipeline dan dataset dengan cache yang dapat dilanjutkan.
3. Ekstrak arsip, gabungkan tiga bagian FishDisease, lalu validasi struktur data.
4. Konversi mask/JSON ke format YOLO dan buat split klasifikasi deterministik.
5. Training tiga model dengan parameter yang dapat diubah.
6. Evaluasi, kurva training, confusion matrix, dashboard, dan laporan.

> Jalankan sel secara berurutan. Untuk menghemat storage, arsip unduhan dihapus setelah ekstraksi dan dataset tidak diunduh ulang jika sudah tersedia.

In [ ]:
# CELL 2 - Instalasi library dan konfigurasi runtime
%pip install -q ultralytics split-folders opencv-python-headless pandas matplotlib seaborn scikit-learn gdown requests

from pathlib import Path
import hashlib
import json
import os
import shutil
import sys
import tarfile
import urllib.request
import zipfile
import platform
import torch
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image, Markdown

# Semua artefak runtime disimpan di /content, bukan di repository notebook.
WORKSPACE = Path('/content/yolocomvis_runtime')
WORKSPACE.mkdir(parents=True, exist_ok=True)

# URL langsung atau Google Drive share link. FishDisease terdiri dari tiga arsip.
DATASET_URLS = {
    'fish4knowledge': 'https://homepages.inf.ed.ac.uk/rbf/Fish4Knowledge/GROUNDTRUTH/RECOG/Archive/fishRecognition_GT.tar',
    'fishdisease_mentah': [
        'https://drive.google.com/file/d/1sFotl0luJmJ0eB2v5o_uoieczo6fzK8U/view',
        'https://drive.google.com/file/d/1ZCEzHHPvbeQBgKH9x6NuygD2m5Cv3ftX/view',
        'https://drive.google.com/file/d/1Atq0N96EjVBnX5DJMTAHETZFTSf_13UF/view',
    ],
    'freshwater_asli': 'https://www.kaggle.com/api/v1/datasets/download/subirbiswas19/freshwater-fish-disease-aquaculture-in-south-asia',
}

# URL raw, bukan URL halaman HTML GitHub.
PIPELINE_URL = 'https://raw.githubusercontent.com/F4-dly/comviskan/main/all_in_one_yolocomvis.py'
PROJECT_BUNDLE_URL = ''  # Opsional: isi hanya jika ingin memakai satu ZIP proyek.

SEED = 42
EPOCHS = 10
IMAGE_SIZE_DETECTION = 640
IMAGE_SIZE_CLASSIFICATION = 224
VAL_RATIO = 0.20
FORCE_REDOWNLOAD = False
KEEP_ARCHIVES = False

# Pipeline kecil selalu diambil ulang agar perbaikan terbaru dipakai.
pipeline_cache = WORKSPACE / 'all_in_one_yolocomvis.py'
if pipeline_cache.exists():
    pipeline_cache.unlink()
    print('Cache pipeline lama dihapus; script Python terbaru akan diunduh ulang.')

TRAIN_PROJECT = str(WORKSPACE / 'Runs_Baseline')
print(f'Python    : {platform.python_version()}')
print(f'PyTorch   : {torch.__version__}')
print(f'GPU aktif : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU name  : {torch.cuda.get_device_name(0)}')
print(f'Workspace : {WORKSPACE}')
print(f'Epoch     : {EPOCHS} | seed: {SEED} | val: {VAL_RATIO:.0%}')

## 1. Download otomatis dataset dan kode

Sel berikut membaca URL pada sel 2, mengunduh arsip, mengekstrak folder dataset, menyiapkan bobot YOLO, dan mengimpor pipeline. Untuk pengalaman paling mandiri, gunakan satu `PROJECT_BUNDLE_URL` yang berisi `all_in_one_yolocomvis.py`, bobot `.pt`, dan folder `datasets/`.

In [ ]:
# CELL 3 - Download, ekstraksi, validasi, dan import pipeline
import requests


def _as_url_list(value):
    if isinstance(value, (list, tuple)):
        return [url for url in value if url]
    return [value] if value else []


def download_file(url, destination):
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.exists() and destination.stat().st_size > 0 and not FORCE_REDOWNLOAD:
        print(f'Cache dipakai: {destination.name}')
        return destination

    if 'drive.google.com' in url:
        import gdown
        result = gdown.download(url, str(destination), quiet=False, fuzzy=True)
        if result is None:
            raise RuntimeError(f'Google Drive gagal diunduh: {url}')
    else:
        with requests.get(url, stream=True, timeout=120) as response:
            response.raise_for_status()
            with destination.open('wb') as file:
                for chunk in response.iter_content(chunk_size=8 * 1024 * 1024):
                    if chunk:
                        file.write(chunk)

    if not destination.exists() or destination.stat().st_size == 0:
        raise RuntimeError(f'File hasil download kosong: {url}')
    print(f'Download selesai: {destination.name} ({destination.stat().st_size / 1024**2:.1f} MB)')
    return destination


def _safe_member_path(root, member_name):
    root = Path(root).resolve()
    target = (root / member_name).resolve()
    if target != root and root not in target.parents:
        raise ValueError(f'Path arsip tidak aman: {member_name}')
    return target


def extract_archive(archive_path, destination):
    archive_path = Path(archive_path)
    destination = Path(destination)
    destination.mkdir(parents=True, exist_ok=True)
    if zipfile.is_zipfile(archive_path):
        with zipfile.ZipFile(archive_path) as archive:
            for member in archive.infolist():
                _safe_member_path(destination, member.filename)
            archive.extractall(destination)
    elif tarfile.is_tarfile(archive_path):
        with tarfile.open(archive_path) as archive:
            for member in archive.getmembers():
                _safe_member_path(destination, member.name)
            archive.extractall(destination)
    else:
        raise ValueError(f'Arsip tidak didukung atau URL bukan file arsip: {archive_path}')


def find_directory(root, names):
    for name in names:
        direct = root / name
        if direct.is_dir():
            return direct
    for name in names:
        matches = list(root.rglob(name))
        if matches:
            return matches[0]
    return None


def _has_extracted_content(path):
    return path.exists() and any(path.iterdir())


def prepare_downloaded_data():
    urls = dict(DATASET_URLS)
    if PROJECT_BUNDLE_URL:
        urls['project_bundle'] = PROJECT_BUNDLE_URL
    active = {key: value for key, value in urls.items() if _as_url_list(value)}
    if not active:
        raise ValueError('Isi minimal satu URL dataset atau PROJECT_BUNDLE_URL di Cell 2.')

    download_dir = WORKSPACE / '_downloads'
    extract_dir = WORKSPACE / '_extracted'
    download_dir.mkdir(exist_ok=True)
    extract_dir.mkdir(exist_ok=True)

    for key, source in active.items():
        destination = extract_dir / key
        if _has_extracted_content(destination) and not FORCE_REDOWNLOAD:
            print(f'Ekstraksi cache dipakai: {key}')
            continue
        for index, url in enumerate(_as_url_list(source), start=1):
            archive = download_file(url, download_dir / f'{key}_{index}.archive')
            extract_archive(archive, destination)
            if not KEEP_ARCHIVES:
                archive.unlink(missing_ok=True)

    search_root = extract_dir / 'project_bundle' if (extract_dir / 'project_bundle').exists() else extract_dir
    mapping = {
        'fish4knowledge': ['fish4knowledge', 'fishRecognition_GT'],
        'fishdisease_mentah': ['fishdisease_mentah'],
        'freshwater_asli': ['freshwater_asli', 'freshwater-fish-disease-aquaculture-in-south-asia'],
    }
    for target_name, candidates in mapping.items():
        target = WORKSPACE / 'datasets' / target_name
        if target.exists() and not FORCE_REDOWNLOAD:
            print(f'Dataset sudah ada: {target_name}')
            continue
        source = find_directory(search_root, candidates)
        if source is None:
            direct_source = extract_dir / target_name
            source = direct_source if _has_extracted_content(direct_source) else None
        if source is None:
            print(f'Peringatan: folder {target_name} tidak ditemukan dari URL yang diisi.')
            continue
        if target.exists():
            shutil.rmtree(target)
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copytree(source, target)
        print(f'{target_name}: {target}')

    pipeline_file = WORKSPACE / 'all_in_one_yolocomvis.py'
    if not pipeline_file.exists() or FORCE_REDOWNLOAD:
        if PIPELINE_URL:
            download_file(PIPELINE_URL, pipeline_file)
    for filename in ('all_in_one_yolocomvis.py', 'yolo11n.pt', 'yolo11n-cls.pt'):
        matches = list(search_root.rglob(filename))
        target = WORKSPACE / filename
        if matches and (filename != 'all_in_one_yolocomvis.py' or not target.exists()):
            shutil.copy2(matches[0], target)

    if not KEEP_ARCHIVES:
        shutil.rmtree(download_dir, ignore_errors=True)
    shutil.rmtree(extract_dir, ignore_errors=True)


prepare_downloaded_data()

# Ambil bobot resmi Ultralytics bila belum ada di workspace.
from ultralytics import YOLO
for weight_name in ('yolo11n.pt', 'yolo11n-cls.pt'):
    target = WORKSPACE / weight_name
    if not target.exists():
        YOLO(weight_name)
        downloaded = Path(weight_name)
        if downloaded.exists():
            shutil.copy2(downloaded, target)

pipeline_file = WORKSPACE / 'all_in_one_yolocomvis.py'
if not pipeline_file.exists():
    raise FileNotFoundError('Pipeline tidak tersedia. Periksa PIPELINE_URL atau PROJECT_BUNDLE_URL.')
sys.path.insert(0, str(WORKSPACE))
from all_in_one_yolocomvis import (
    dataset_summary, save_dataset_report, prepare_fish_dataset,
    prepare_lesion_dataset, prepare_classification_dataset, train_models,
    evaluate_models, plot_training_curves, classification_confusion_matrix,
    write_final_report, run_dashboard,
)
print('Dataset, pipeline, dan bobot siap.')

In [ ]:
# CELL 3B - Normalisasi layout arsip Kaggle dan pemeriksaan minimum

def normalize_freshwater_layout():
    root = WORKSPACE / 'datasets' / 'freshwater_asli'
    train_dir = root / 'Train'
    if not train_dir.exists():
        candidates = list(root.rglob('Train')) if root.exists() else []
        train_dir = next((path for path in candidates if path.is_dir()), None)
    if train_dir is None or not train_dir.exists():
        return

    for class_dir in sorted(path for path in train_dir.iterdir() if path.is_dir()):
        destination = root / class_dir.name
        if not destination.exists():
            shutil.move(str(class_dir), str(destination))
    test_dir = train_dir.parent / 'Test'
    shutil.rmtree(train_dir, ignore_errors=True)
    shutil.rmtree(test_dir, ignore_errors=True)
    print('Layout klasifikasi Kaggle dinormalisasi ke folder kelas langsung.')


def validate_raw_datasets():
    checks = {
        'fish4knowledge': WORKSPACE / 'datasets' / 'fish4knowledge',
        'fishdisease_mentah': WORKSPACE / 'datasets' / 'fishdisease_mentah',
        'freshwater_asli': WORKSPACE / 'datasets' / 'freshwater_asli',
        'pipeline': WORKSPACE / 'all_in_one_yolocomvis.py',
    }
    rows = []
    for name, path in checks.items():
        files = sum(1 for item in path.rglob('*') if item.is_file()) if path.exists() else 0
        rows.append({'komponen': name, 'path': str(path), 'ada': path.exists(), 'file': files})
    result = pd.DataFrame(rows)
    display(result)
    missing = result.loc[~result['ada'], 'komponen'].tolist()
    if missing:
        raise FileNotFoundError(f'Komponen belum tersedia: {missing}')


normalize_freshwater_layout()
validate_raw_datasets()

In [ ]:
# CELL 4 - Audit awal dataset mentah dan output YOLO
# Pada tahap ini folder images/train dan labels/train memang belum dibuat.
# Karena itu statistik deteksi dapat bernilai 0; jumlah final dihitung ulang pada Cell 5.
raw_checks = {
    'fish4knowledge_raw': WORKSPACE / 'datasets' / 'fish4knowledge',
    'fishdisease_raw': WORKSPACE / 'datasets' / 'fishdisease_mentah',
    'freshwater_raw': WORKSPACE / 'datasets' / 'freshwater_asli',
}
raw_rows = []
for name, path in raw_checks.items():
    file_count = sum(1 for item in path.rglob('*') if item.is_file()) if path.exists() else 0
    raw_rows.append({'dataset_mentah': name, 'path': str(path), 'tersedia': path.exists(), 'jumlah_file': file_count})
display(Markdown('### Pemeriksaan dataset mentah setelah download'))
display(pd.DataFrame(raw_rows))
if not all(row['tersedia'] and row['jumlah_file'] > 0 for row in raw_rows):
    raise FileNotFoundError('Dataset mentah belum lengkap. Periksa kembali hasil Cell 3.')

display(Markdown('### Kondisi folder YOLO sebelum preprocessing'))
summary_before = dataset_summary(WORKSPACE)
display(pd.json_normalize(summary_before, sep='_').T.rename(columns={0: 'nilai'}))
display(Markdown('> Nilai 0 pada bagian deteksi di sini normal. Jalankan Cell 5 untuk membuat label YOLO dan split train/val.'))

## 2. Preprocessing dan konversi anotasi

Tahap ini tidak mengubah arsip sumber:

- **Fish4Knowledge**: mask PNG dikonversi menjadi bounding box YOLO kelas `fish`.
- **FishDisease**: JSON anotasi dibaca dan dicocokkan dengan gambar dari tiga arsip mentah, lalu dikonversi menjadi bounding box YOLO kelas `lesion`.
- **Freshwater Kaggle**: folder `Train/<kelas>` dinormalisasi, lalu dibagi menjadi `train` dan `val` menggunakan seed tetap.

Output preprocessing berada di `datasets/fish4knowledge`, `datasets/fishdisease`, dan `datasets/freshwater_kaggle`. Jika tahap ini dijalankan ulang, hasil split klasifikasi akan dibuat ulang secara deterministik.

In [ ]:
# CELL 5 - Preprocessing dan konversi anotasi
prepare_fish_dataset(WORKSPACE, val_ratio=VAL_RATIO, seed=SEED)
prepare_lesion_dataset(WORKSPACE, val_ratio=VAL_RATIO, seed=SEED)

# Splitfolders membuat output baru; hapus hanya hasil split, bukan data mentah.
classification_output = WORKSPACE / 'datasets' / 'freshwater_kaggle'
if classification_output.exists():
    shutil.rmtree(classification_output)
prepare_classification_dataset(WORKSPACE, seed=SEED)

summary_after = dataset_summary(WORKSPACE)
save_dataset_report(WORKSPACE)

# Jangan lanjut ke training jika folder deteksi belum berisi pasangan valid.
for dataset_name in ('fish_detection', 'lesion_detection'):
    split_stats = summary_after[dataset_name]['splits']
    train_images = split_stats['train']['images']
    train_labels = split_stats['train']['labels']
    val_images = split_stats['val']['images']
    val_labels = split_stats['val']['labels']
    if min(train_images, train_labels, val_images, val_labels) == 0:
        raise RuntimeError(
            f'{dataset_name} belum siap training: {split_stats}. '
            'Periksa hasil preprocessing sebelum menjalankan Cell 6.'
        )

display(Markdown('### Ringkasan setelah preprocessing'))
for dataset_name in ('fish_detection', 'lesion_detection'):
    rows = [{'dataset': dataset_name, 'split': split, **values} for split, values in summary_after[dataset_name]['splits'].items()]
    display(pd.DataFrame(rows))
display(Markdown('### Distribusi kelas klasifikasi'))
display(pd.DataFrame(summary_after['disease_classification']['classes']).fillna(0).astype(int))

## 3. Training tiga model

Training membuat tiga checkpoint di `Runs_Baseline`:

1. Deteksi ikan dari Fish4Knowledge.
2. Deteksi lesi dari FishDisease.
3. Klasifikasi penyakit dari Freshwater Fish Disease.

Parameter baseline mengikuti eksperimen awal. Ubah `EPOCHS` pada Cell 2 untuk eksperimen lebih panjang. Pastikan GPU aktif sebelum memulai karena training dapat memerlukan waktu dan storage cukup besar.

In [ ]:
# CELL 6 - Training tiga model
training_results = train_models(WORKSPACE, epochs=EPOCHS, project=TRAIN_PROJECT)
print('Training selesai. Checkpoint best.pt tersimpan di:', TRAIN_PROJECT)

## 4. Evaluasi kuantitatif

Sel berikut menjalankan validasi pada checkpoint terbaik. Untuk deteksi, metrik utama adalah precision, recall, mAP50, dan mAP50-95. Untuk klasifikasi, metrik utama adalah top-1 dan top-5 accuracy.

In [ ]:
# CELL 7 - Evaluasi kuantitatif
evaluation = evaluate_models(WORKSPACE, project=TRAIN_PROJECT)
evaluation_table = pd.DataFrame(evaluation).T.reset_index().rename(columns={'index': 'model'})
display(evaluation_table)
print('File metrik:', WORKSPACE / 'reports' / 'evaluation_metrics.json')

## 5. Kurva training dan confusion matrix

Kurva membantu membaca konvergensi loss serta perubahan precision/recall/mAP. Confusion matrix menunjukkan kelas penyakit yang sering tertukar pada split validasi.

In [ ]:
# CELL 8 - Kurva training dan confusion matrix
curve_paths = plot_training_curves(WORKSPACE, project=TRAIN_PROJECT)
for path in curve_paths:
    display(Markdown(f'### {path.name}'))
    display(Image(filename=str(path)))

confusion_path = classification_confusion_matrix(WORKSPACE, project=TRAIN_PROJECT)
if confusion_path:
    display(Image(filename=str(confusion_path)))

## 6. Dashboard inferensi

Dashboard menggabungkan tiga tahap: deteksi ikan, deteksi lesi pada crop ikan, dan top-3 klasifikasi penyakit. Gambar disimpan sebagai artefak PNG/JPG sehingga dapat dimasukkan ke laporan.

In [ ]:
# CELL 9 - Dashboard inferensi
TEST_IMAGES = [WORKSPACE / name for name in ('ujicoba.png', 'ujicoba2.png', 'ujicoba10.png')]
if not any(path.exists() for path in TEST_IMAGES):
    fish_images = sorted((WORKSPACE / 'datasets' / 'fish4knowledge' / 'images').rglob('*'))
    TEST_IMAGES = [path for path in fish_images if path.suffix.lower() in ('.jpg', '.jpeg', '.png', '.bmp', '.webp')][:3]
if not TEST_IMAGES:
    print('Dashboard dilewati: tidak ada gambar uji yang tersedia.')
for test_image in TEST_IMAGES:
    if not test_image.exists():
        continue
    output_image = WORKSPACE / 'reports' / f'dashboard_{test_image.stem}.jpg'
    run_dashboard(WORKSPACE, test_image, output_image)
    display(Markdown(f'### Hasil inferensi: `{test_image.name}`'))
    display(Image(filename=str(output_image)))

## 7. Laporan akhir dan artefak

Laporan Markdown merangkum konfigurasi, statistik dataset, dan metrik evaluasi. File JSON/CSV cocok untuk analisis lanjutan, sedangkan PNG cocok untuk dimasukkan ke bab hasil.

In [ ]:
# CELL 10 - Laporan akhir dan artefak
final_report = write_final_report(WORKSPACE, dataset_info=summary_after, evaluation=evaluation)
print(final_report.read_text(encoding='utf-8'))

report_files = sorted((WORKSPACE / 'reports').glob('*'))
display(pd.DataFrame({'artefak': [path.name for path in report_files], 'ukuran_kb': [round(path.stat().st_size / 1024, 2) for path in report_files]}))